In [2]:
import pandas as pd
from pathlib import Path
import numpy as np

run_dir = Path(
    "/projects/e32998/MIL_training/final_runs/run_20261103_101400"
)

csv_path = (
    run_dir / "attention_analysis"
    / "top_effective_patches_per_case_5.0pct.csv"
)

png_dir = Path("/projects/e32998/patches_benign_split")
pt_dir = Path("/projects/e32998/patches_varsize_pooled4096")

df = pd.read_csv(csv_path)

print("Total attention records:", len(df))
print("Cases:", df["case_id"].nunique())

# Check every saved record has a real PNG and embedding
missing_pngs = []
missing_pts = []

for filename in df["patch_filename"]:
    png = png_dir / filename
    pt = pt_dir / (Path(filename).stem + ".pt")

    if not png.is_file():
        missing_pngs.append(filename)

    if not pt.is_file():
        missing_pts.append(filename)

print("Missing PNGs:", len(missing_pngs))
print("Missing embeddings:", len(missing_pts))

# Verify effective attention calculation
expected = (
    df["patch_attn_weight"]
    * df["slice_attn_weight"]
    * df["stain_attn_weight"]
)

matches = np.allclose(
    df["effective_weight"],
    expected,
    rtol=1e-5,
    atol=1e-8
)

print("Effective attention calculation correct:", matches)

assert len(missing_pngs) == 0
assert len(missing_pts) == 0
assert matches

print("Saved attention records passed verification!")


Total attention records: 128
Cases: 19
Missing PNGs: 0
Missing embeddings: 0
Effective attention calculation correct: True
Saved attention records passed verification!


In [4]:

import tempfile
from pathlib import Path
import torch

from dataset import StainBagCasePooledFeatureDataset

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    paths = []

    for i in range(20):
        filename = f"case_1_match_1_h&e_patch{i}.png"
        path = root / filename

        path.touch()
        torch.save(
            torch.full((4096,), float(i)),
            root / path.with_suffix(".pt").name
        )
        paths.append(str(path))

    ds = StainBagCasePooledFeatureDataset(
        case_dict={
            1: {"h&e": [paths]}
        },
        label_map={1: 0},
        embeddings_dir=str(root),
        stains=("h&e",),
        per_slice_cap=20,
        shuffle_patches=True
    )

    for _ in range(10):
        sample = ds[0]

        features = sample["stain_slices"]["h&e"][0]
        filenames = sample["patch_filenames"]["h&e"][0]

        assert len(features) == len(filenames)

        for feature, filename in zip(features, filenames):
            patch_num = int(
                filename.split("_patch")[-1].split(".")[0]
            )

            assert torch.all(
                feature == float(patch_num)
            ), f"Mismatch for {filename}"

print("PASS: Filenames stay aligned with embeddings after shuffling!")


PASS: Filenames stay aligned with embeddings after shuffling!
